# 🦺 PPE Compliance Detection — YOLO11 on Construction-PPE
**Computer Vision Systems Development · Final Project**

This notebook is the full pipeline: **data → preprocessing/augmentation → training → evaluation → success/failure analysis → compliance logic → ONNX export → INT8 quantization → benchmark**.

> Run on **Google Colab** with a GPU: *Runtime → Change runtime type → T4 GPU*. The full run takes about 30–40 minutes.

## 1. Problem Definition
Safety inspections on construction and industrial sites are manual, intermittent, and cannot cover every camera at all times. Missing helmets and high-visibility vests are among the most common causes of serious site injuries.

**Goal:** build a computer vision system that watches a camera image or video frame and automatically flags workers who are not wearing required PPE.

- **Input:** an image or video frame from a site camera.
- **CV task:** object detection (YOLO11n, fine-tuned with transfer learning).
- **Output:** bounding boxes for `Person`, `helmet`, `vest`, `no_helmet` and other PPE classes, plus a **per-person compliance decision** and a **frame status** (`COMPLIANT` / `VIOLATION` / `NO PERSON`).

## 2. Setup

In [ ]:
# If running on Colab, clone your repo first (edit REPO_URL to your GitHub repo)
import os, sys
REPO_URL = "https://github.com/<your-username>/ppe-detection.git"
if "google.colab" in sys.modules and not os.path.exists("src/ppe_utils.py"):
    !git clone -q {REPO_URL} /content/ppe-detection
    %cd /content/ppe-detection

!pip install -q "ultralytics>=8.3.0" onnx onnxruntime onnxslim tabulate
sys.path.insert(0, "src")

In [ ]:
import glob, json, shutil, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

from ppe_utils import (check_compliance, detections_from_result, image_f1,
                       load_yolo_labels, quantize_onnx_int8)

# ---- configuration ----
DATA     = "construction-ppe.yaml"   # built into Ultralytics, auto-downloads (~178 MB)
BASE     = "yolo11n.pt"              # COCO-pretrained weights -> transfer learning
EPOCHS   = 60
IMGSZ    = 640
BATCH    = 16
RUN_NAME = "ppe_yolo11n"
ASSETS, MODELS = Path("assets"), Path("models")
ASSETS.mkdir(exist_ok=True); MODELS.mkdir(exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU (training will be slow)")

## 3. Dataset
**Construction-PPE** (Ultralytics) contains real construction-site photos annotated in YOLO format, pre-split into **train 1,132 / val 143 / test 141** images.

11 classes: `helmet, gloves, vest, boots, goggles, none, Person, no_helmet, no_goggle, no_gloves, no_boots`.

It was chosen because it includes both **positive** (gear present) and **negative** (`no_helmet`, …) classes, which is exactly what a compliance system needs. Annotation is already done, so no manual labeling was required. The same pipeline works with any Roboflow PPE dataset exported in *YOLOv8* format: just point `DATA` at its `data.yaml`.

In [ ]:
info = check_det_dataset(DATA)          # downloads the dataset if needed
names = info["names"]
print("Classes:", names)

def label_dir(img_dir):
    return str(img_dir).replace(f"{os.sep}images", f"{os.sep}labels")

counts = {}
for split in ["train", "val", "test"]:
    c = np.zeros(len(names), dtype=int)
    for f in glob.glob(os.path.join(label_dir(info[split]), "*.txt")):
        for line in open(f):
            if line.strip():
                c[int(line.split()[0])] += 1
    counts[split] = c
dist = pd.DataFrame(counts, index=[names[i] for i in range(len(names))])
display(dist)

dist["train"].sort_values().plot.barh(figsize=(7, 4), title="Train instances per class")
plt.tight_layout(); plt.savefig(ASSETS / "class_distribution.png", dpi=120); plt.show()

In [ ]:
# Visualize a few training images with their ground-truth boxes
rng = np.random.default_rng(0)
imgs = sorted(glob.glob(os.path.join(info["train"], "*.jpg")))
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, p in zip(axes.ravel(), rng.choice(imgs, 6, replace=False)):
    im = cv2.imread(p); h, w = im.shape[:2]
    lbl = os.path.join(label_dir(info["train"]), Path(p).stem + ".txt")
    for c, (x1, y1, x2, y2) in load_yolo_labels(lbl, w, h):
        cv2.rectangle(im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
        cv2.putText(im, names[c], (int(x1), int(y1) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    ax.imshow(im[:, :, ::-1]); ax.axis("off")
plt.tight_layout(); plt.savefig(ASSETS / "samples.png", dpi=100); plt.show()

## 4. Preprocessing & Augmentation
**Preprocessing (applied at train and inference):** letterbox resize to 640×640 (aspect ratio kept, gray padding), BGR→RGB, scale pixels to 0–1.

**Augmentation (training only), chosen for site conditions:**

| Augmentation | Setting | Why |
|---|---|---|
| HSV color jitter | h=0.015, s=0.7, v=0.4 | outdoor lighting, shadows, sun glare |
| Horizontal flip | 0.5 | workers face both directions |
| Scale / translate | 0.5 / 0.1 | workers appear near and far from camera |
| Small rotation | ±5° | tilted or handheld cameras |
| Mosaic | 1.0 (off in last 10 epochs) | more small objects & context per batch |

**Transfer learning:** start from YOLO11n weights pretrained on COCO (which already knows "person"), then fine-tune all layers on PPE classes.

## 5. Training

In [ ]:
model = YOLO(BASE)
model.train(
    data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    patience=15, seed=42, project="runs", name=RUN_NAME, exist_ok=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, fliplr=0.5,
    scale=0.5, translate=0.1, degrees=5.0, mosaic=1.0, close_mosaic=10,
)
RUN_DIR = Path(model.trainer.save_dir)
shutil.copy(RUN_DIR / "weights" / "best.pt", MODELS / "best.pt")
print("Best weights ->", MODELS / "best.pt")

## 6. Evaluation
Metrics on the **held-out test split** (never seen during training or model selection):
- **mAP@50**: mean average precision at IoU ≥ 0.5 (main detection metric)
- **mAP@50-95**: stricter, averaged over IoU 0.50→0.95
- **Precision / Recall** per class: for safety, **recall on `no_helmet`** matters most (a missed violation is worse than a false alarm)

In [ ]:
best = YOLO(MODELS / "best.pt")
m = best.val(data=DATA, split="test", imgsz=IMGSZ, device=DEVICE, plots=True,
             project="runs", name=f"{RUN_NAME}_test", exist_ok=True)

overall = {"precision": float(m.box.mp), "recall": float(m.box.mr),
           "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map)}
print(json.dumps(overall, indent=2))

per_class = pd.DataFrame({
    "class": [names[i] for i in m.box.ap_class_index],
    "precision": m.box.p, "recall": m.box.r,
    "mAP50": m.box.ap50, "mAP50_95": m.box.ap,
}).round(3).sort_values("mAP50", ascending=False)
display(per_class)

per_class.to_csv(ASSETS / "per_class_metrics.csv", index=False)
json.dump(overall, open(ASSETS / "metrics.json", "w"), indent=2)

# copy training/eval plots into assets/ for the README
TEST_DIR = Path(m.save_dir)
for src, dst in [(RUN_DIR / "results.png", "training_curves.png"),
                 (TEST_DIR / "confusion_matrix_normalized.png", "confusion_matrix.png")]:
    if src.exists(): shutil.copy(src, ASSETS / dst)
pr = [p for p in TEST_DIR.glob("*PR_curve.png")]
if pr: shutil.copy(pr[0], ASSETS / "pr_curve.png")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, f in zip(axes, ["training_curves.png", "confusion_matrix.png", "pr_curve.png"]):
    if (ASSETS / f).exists():
        ax.imshow(plt.imread(ASSETS / f)); ax.set_title(f)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7. Success & Failure Cases
Each test image is scored by comparing predictions to ground truth (class-aware matching at IoU ≥ 0.5 → per-image F1). The **best** image with ≥ 3 objects is the success case; the **worst** image with ≥ 2 objects is the failure case.

In [ ]:
test_imgs = sorted(glob.glob(os.path.join(info["test"], "*.jpg")))
rows = []
for p in test_imgs:
    r = best.predict(p, conf=0.25, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    h, w = r.orig_shape
    gts = load_yolo_labels(os.path.join(label_dir(info["test"]), Path(p).stem + ".txt"), w, h)
    preds = [(int(c), float(s), tuple(b)) for c, s, b in
             zip(r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.xyxy.cpu().numpy())]
    rows.append({"path": p, "n_gt": len(gts), **image_f1(preds, gts)})
scores = pd.DataFrame(rows)

success = scores[scores.n_gt >= 3].sort_values(["f1", "n_gt"], ascending=[False, False]).iloc[0]
failure = scores[scores.n_gt >= 2].sort_values(["f1", "n_gt"], ascending=[True, False]).iloc[0]
display(pd.DataFrame([success, failure], index=["success", "failure"]))

def side_by_side(row, out_name, title):
    im = cv2.imread(row.path); h, w = im.shape[:2]
    gt_im = im.copy()
    for c, (x1, y1, x2, y2) in load_yolo_labels(
            os.path.join(label_dir(info["test"]), Path(row.path).stem + ".txt"), w, h):
        cv2.rectangle(gt_im, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 2)
        cv2.putText(gt_im, names[c], (int(x1), int(y1) - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
    pred_im = best.predict(row.path, conf=0.25, imgsz=IMGSZ, device=DEVICE, verbose=False)[0].plot()
    fig, ax = plt.subplots(1, 2, figsize=(16, 7))
    ax[0].imshow(gt_im[:, :, ::-1]); ax[0].set_title("Ground truth")
    ax[1].imshow(pred_im[:, :, ::-1]); ax[1].set_title(f"Prediction  (TP={row.tp}, FP={row.fp}, FN={row.fn})")
    for a in ax: a.axis("off")
    fig.suptitle(title, fontsize=15); plt.tight_layout()
    plt.savefig(ASSETS / out_name, dpi=110); plt.show()

side_by_side(success, "success_case.png", "✅ Success case")
side_by_side(failure, "failure_case.png", "❌ Failure case")

**Failure analysis (write your observations here after running):** look at the failure image and note *why* it failed. Common causes on this dataset:
- small or distant workers → missed boxes (false negatives)
- occlusion (worker behind equipment or another worker)
- look-alike objects (caps, hoods, orange clothing confused with helmet/vest)
- rare classes with few training examples (see class distribution above)

## 8. From Detections to a Compliance Decision
Raw boxes are not yet a useful safety output. `check_compliance()` assigns each PPE box to the person box that contains most of it, then marks a person **non-compliant** if a required item (default: **helmet + vest**) is missing or a `no_helmet` box is assigned to them.

In [ ]:
def compliance_image(path, required=("helmet", "vest")):
    r = best.predict(path, conf=0.25, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    persons, summary = check_compliance(detections_from_result(r), required)
    im = r.plot()
    for s in persons:
        x1, y1, x2, y2 = map(int, s.box)
        color = (0, 200, 0) if s.compliant else (0, 0, 255)
        label = "OK" if s.compliant else "MISSING: " + ", ".join(s.missing)
        cv2.rectangle(im, (x1, y1), (x2, y2), color, 4)
        cv2.putText(im, label, (x1, y2 + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return im, summary

im, summary = compliance_image(success.path)
print(json.dumps(summary, indent=2))
plt.figure(figsize=(10, 7)); plt.imshow(im[:, :, ::-1]); plt.axis("off")
plt.title(f"Frame status: {summary['frame_status']}")
plt.savefig(ASSETS / "compliance_demo.png", dpi=110, bbox_inches="tight"); plt.show()

## 9. Deployment & Optimization: ONNX + INT8 Quantization
1. **ONNX export**: framework-independent model that runs with ONNX Runtime on CPU, edge devices, or servers without PyTorch.
2. **INT8 static quantization** (ONNX Runtime, QDQ format, per-channel): weights and activations stored as 8-bit integers, calibrated on 100 training images. The detection head is kept in FP32 to protect box accuracy.
3. **Benchmark** all three variants on CPU (a realistic edge / on-prem server setting) on the test set.

In [ ]:
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False)
# best.pt lives in models/, so export already writes models/best.onnx -> copy only if different
if Path(onnx_path).resolve() != (MODELS / "best.onnx").resolve():
    shutil.copy(onnx_path, MODELS / "best.onnx")

quantize_onnx_int8(str(MODELS / "best.onnx"), str(MODELS / "best_int8.onnx"),
                   calib_image_dir=info["train"], n_calib=100, size=IMGSZ)

In [ ]:
variants = {"PyTorch FP32": MODELS / "best.pt",
            "ONNX FP32":    MODELS / "best.onnx",
            "ONNX INT8":    MODELS / "best_int8.onnx"}
bench = []
for name, path in variants.items():
    mdl = YOLO(str(path), task="detect")
    r = mdl.val(data=DATA, split="test", imgsz=IMGSZ, batch=1, device="cpu",
                plots=False, verbose=False, project="runs", name="bench", exist_ok=True)
    bench.append({"model": name,
                  "size_MB": round(os.path.getsize(path) / 1e6, 2),
                  "mAP50": round(float(r.box.map50), 3),
                  "mAP50_95": round(float(r.box.map), 3),
                  "cpu_ms_per_img": round(r.speed["inference"], 1)})
bench = pd.DataFrame(bench)
bench["fps"] = (1000 / bench.cpu_ms_per_img).round(1)
display(bench)
bench.to_csv(ASSETS / "benchmark.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
bench.plot.bar(x="model", y="cpu_ms_per_img", ax=ax[0], legend=False, title="CPU latency (ms/img, lower is better)", rot=0)
bench.plot.bar(x="model", y="mAP50", ax=ax[1], legend=False, title="Test mAP@50", rot=0, ylim=(0, 1))
plt.tight_layout(); plt.savefig(ASSETS / "benchmark.png", dpi=120); plt.show()

In [ ]:
# Write a markdown results summary you can paste straight into README.md
with open(ASSETS / "results.md", "w") as f:
    f.write("### Test-set metrics\n\n")
    f.write(pd.DataFrame([overall]).round(3).to_markdown(index=False) + "\n\n")
    f.write("### Per-class metrics\n\n" + per_class.to_markdown(index=False) + "\n\n")
    f.write("### Optimization benchmark (CPU)\n\n" + bench.to_markdown(index=False) + "\n")
print(open(ASSETS / "results.md").read())

## 10. Real Application Demo
Run the web app (image + video upload, live compliance status):
```bash
python app/app.py            # uses models/best.onnx by default
```
Or batch inference from the command line:
```bash
python src/predict.py --source path/to/images_or_video --model models/best_int8.onnx
```

In [ ]:
# Optional: launch the Gradio demo directly from Colab (gives a public share link)
!pip install -q gradio
!python app/app.py --share

## 11. How the Model Could Be Improved
- **More and more-diverse data:** add Saudi / Gulf site footage (desert light, dust, heat haze) and night shots; collect hard negatives (caps, hoods, orange clothing).
- **Class balance:** oversample or collect more examples of rare classes; consider merging low-value classes.
- **Small objects:** train at 960–1280 px or use tiling (SAHI) for distant workers.
- **Bigger model:** YOLO11s/m for accuracy if hardware allows; keep n + INT8 for edge.
- **Temporal logic:** track workers across frames (ByteTrack) and raise an alert only if a violation persists for N seconds → fewer false alarms.
- **Two-stage approach:** detect persons, then classify helmet/vest on each person crop.